---

- Author: Jaelin Lee
- Date: Feb 8, 2026
- Description: Comprehensive LLM-based behavioral analysis of agent session logs with ORPDA layer validation
- Input: Cleaned CSV logs from `app/logs/cleaned/` (39 sessions, ORPA/ORPDA modes)
- Output: Individual + global comparative analyses in `app/logs/analysis_results_LaaJ/`

- Analysis:
  - Layer-by-layer ORPDA functional validation
  - Plan-Action alignment (explicit + implicit)
  - Drift patterns (explicit + implicit)
  - Cross-model and temperature comparison
  - Neuroscience grounding

- Status: 39/39 sessions analyzed

---

## Quick Start Guide

**To run analysis:**

1. **Section 2**: Set filters (model, mode, agent, temperature)
2. **Section 6**: Run batch analysis (individual results auto-save in realtime)
3. **Section 8**: View results (global + individual analyses)

**Output**:
- `GLOBAL_COMPARATIVE_ANALYSIS_[timestamp].txt` - Cross-session patterns
- `individual_*.txt` - Per-session analysis

**Config** (Section 5):
- `RUN_BATCH_ANALYSIS = True`
- `SAVE_REALTIME = True` (auto-saves during processing)

In [1]:
!pip install ipywidgets

## 0. Import

In [2]:
import os
import sys
import asyncio
from pathlib import Path
from pprint import pprint
import pandas as pd
from warnings import filterwarnings
filterwarnings("ignore")

# For rendering markdown in notebooks
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## 1. Setup Paths & Load Available Logs

In [3]:
ROOT = Path.cwd().parents[0]
CLEANED_PATH = Path(ROOT, 'app/logs/cleaned/')
CLEANED_WORKING_PATH = Path(ROOT, 'app/logs/cleaned_working/')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT: {ROOT}")
print(f"CLEANED_PATH: {CLEANED_PATH}")
print(f"CLEANED_WORKING_PATH: {CLEANED_WORKING_PATH}")

ROOT: /Users/jaelinlee/Driftville_Agent
CLEANED_PATH: /Users/jaelinlee/Driftville_Agent/app/logs/cleaned
CLEANED_WORKING_PATH: /Users/jaelinlee/Driftville_Agent/app/logs/cleaned_working


In [4]:
def get_cleaned_logs(path):
    """Get all cleaned CSV files from specified path."""
    csv_files = sorted([f for f in path.glob("cleaned_*.csv")], 
                      key=lambda x: x.stat().st_mtime, reverse=True)
    return csv_files

def parse_log_metadata(filename):
    """Parse metadata from cleaned log CSV content.
    
    Expected format: cleaned_session_orpda_20260207_180031_gpt-oss-20b-cloud_0.0_hailey.csv
    Returns: dict with mode, timestamp, model, temperature, agent
    """
    parts = filename.stem.replace('cleaned_session_', '').split('_')
    
    try:
        # Parse mode from filename
        mode = parts[0] if parts[0] in ['orpa', 'orpda'] else 'unknown'
        
        # Read actual data from CSV
        try:
            df_temp = pd.read_csv(filename, nrows=1)
            agent = df_temp['agent'].iloc[0] if 'agent' in df_temp.columns else 'unknown'
            model = df_temp['llm_model'].iloc[0] if 'llm_model' in df_temp.columns else 'unknown'
            temperature = str(df_temp['temp'].iloc[0]) if 'temp' in df_temp.columns else 'unknown'
            
            # Get timestamp from datetime_start column
            if 'datetime_start_a' in df_temp.columns:
                timestamp = str(df_temp['datetime_start_a'].iloc[0])
            elif 'datetime_start_o' in df_temp.columns:
                timestamp = str(df_temp['datetime_start_o'].iloc[0])
            else:
                timestamp = 'unknown'
                
        except Exception as e:
            print(f"Error reading CSV {filename.name}: {e}")
            agent = 'unknown'
            model = 'unknown'
            temperature = 'unknown'
            timestamp = 'unknown'
            
        return {
            'mode': mode,
            'timestamp': timestamp,
            'model': model,
            'temperature': temperature,
            'agent': agent,
            'filepath': filename
        }
    except Exception as e:
        print(f"Error parsing {filename.name}: {e}")
        return {
            'mode': 'unknown',
            'timestamp': 'unknown',
            'model': 'unknown',
            'temperature': 'unknown',
            'agent': 'unknown',
            'filepath': filename
        }

# Get logs from both directories
cleaned_logs = get_cleaned_logs(CLEANED_PATH)
working_logs = get_cleaned_logs(CLEANED_WORKING_PATH)

print(f"\nFound {len(cleaned_logs)} logs in cleaned/")
print(f"Found {len(working_logs)} logs in cleaned_working/")

# Parse metadata for all logs
all_logs = cleaned_logs + working_logs
log_metadata = [parse_log_metadata(f) for f in all_logs]
logs_df = pd.DataFrame(log_metadata)

print(f"\nTotal logs available: {len(logs_df)}")
display(logs_df.head(10))


Found 77 logs in cleaned/
Found 0 logs in cleaned_working/

Total logs available: 77


,mode,timestamp,model,temperature,agent,filepath
0,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv
1,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.3,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey.csv
2,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.3,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_132744_gemini-3-flash-preview-cloud_0.3_hailey.csv
3,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_143340_gemini-3-flash-preview-cloud_0.0_hailey.csv
4,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_161226_gemini-3-flash-preview-cloud_0.8_hailey.csv
5,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_091642_gemma3-27b-cloud_0.0_hailey.csv
6,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,1.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_092921_gemma3-27b-cloud_1.0_hailey.csv
7,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_094642_gemma3-27b-cloud_0.8_hailey.csv
8,orpa,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_100206_cogito-2.1-671b-cloud_1.0_hailey.csv
9,orpa,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_113455_cogito-2.1-671b-cloud_0.0_hailey.csv


## 2. Filter Options

Set filter criteria (set to `None` to include all)

In [5]:
# Show available values for each filter
print("Available Filters:")
print("="*50)
print(f"\nAgents: {sorted(logs_df['agent'].unique())}")
print(f"\nModels: {sorted(logs_df['model'].unique())}")
print(f"\nModes: {sorted(logs_df['mode'].unique())}")
print(f"\nTemperatures: {sorted(logs_df['temperature'].unique())}")
print(f"\nTimestamp range: {logs_df['timestamp'].min()} to {logs_df['timestamp'].max()}")

Available Filters:

Agents: ['Hailey Johnson', 'Isabella Rodriguez', 'Maria Lopez', 'Sam Moore']

Models: ['cogito-2.1:671b-cloud', 'gemini-3-flash-preview:cloud', 'gemma3:27b-cloud', 'gpt-oss:20b-cloud', 'mistral-large-3:675b-cloud', 'qwen3-next:80b-cloud']

Modes: ['orpa', 'orpda']

Temperatures: ['0.0', '0.2', '0.3', '0.5', '0.8', '1.0']

Timestamp range: 2023-02-13 05:00:00 to 2023-02-13 10:00:00


In [6]:
#########################################
#>> UPDATE FILTERS (None = include all)
FILTER_AGENT = None  # e.g., 'hailey', 'maria', None
FILTER_MODEL = None  # e.g., 'gemini-3-flash-preview:cloud', 'gpt-oss:20b-cloud', None
FILTER_MODE = None  # e.g., 'orpa', 'orpda', None
FILTER_TEMP = None   # e.g., '0.0', '0.8', '1.0', None
FILTER_TIMESTAMP_START = None  # e.g., '20260207_000000', None
FILTER_TIMESTAMP_END = None    # e.g., '20260208_235959', None
#########################################

# Apply filters
filtered_logs = logs_df.copy()

if FILTER_AGENT:
    filtered_logs = filtered_logs[filtered_logs['agent'].str.contains(FILTER_AGENT, case=False, na=False)]
    
if FILTER_MODEL:
    filtered_logs = filtered_logs[filtered_logs['model'].str.contains(FILTER_MODEL, case=False, na=False)]
    
if FILTER_MODE:
    filtered_logs = filtered_logs[filtered_logs['mode'] == FILTER_MODE.lower()]
    
if FILTER_TEMP:
    filtered_logs = filtered_logs[filtered_logs['temperature'] == FILTER_TEMP]
    
if FILTER_TIMESTAMP_START:
    filtered_logs = filtered_logs[filtered_logs['timestamp'] >= FILTER_TIMESTAMP_START]
    
if FILTER_TIMESTAMP_END:
    filtered_logs = filtered_logs[filtered_logs['timestamp'] <= FILTER_TIMESTAMP_END]

print(f"\nFiltered to {len(filtered_logs)} logs")
print("="*50)
display(filtered_logs)


Filtered to 77 logs


,mode,timestamp,model,temperature,agent,filepath
0,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv
1,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.3,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey.csv
2,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.3,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_132744_gemini-3-flash-preview-cloud_0.3_hailey.csv
3,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_143340_gemini-3-flash-preview-cloud_0.0_hailey.csv
4,orpa,2023-02-13 10:00:00,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260206_161226_gemini-3-flash-preview-cloud_0.8_hailey.csv
5,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_091642_gemma3-27b-cloud_0.0_hailey.csv
6,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,1.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_092921_gemma3-27b-cloud_1.0_hailey.csv
7,orpa,2023-02-13 10:00:00,gemma3:27b-cloud,0.8,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_094642_gemma3-27b-cloud_0.8_hailey.csv
8,orpa,2023-02-13 10:00:00,cogito-2.1:671b-cloud,1.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_100206_cogito-2.1-671b-cloud_1.0_hailey.csv
9,orpa,2023-02-13 10:00:00,cogito-2.1:671b-cloud,0.0,Hailey Johnson,/Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpa_20260207_113455_cogito-2.1-671b-cloud_0.0_hailey.csv


## 3. Load & Explore Selected Logs

In [7]:
# Select which log to analyze (by index)
LOG_INDEX = 0  # Change this to select different log

if len(filtered_logs) == 0:
    print("No logs match the current filters!")
else:
    selected_log = filtered_logs.iloc[LOG_INDEX]
    selected_path = selected_log['filepath']
    
    print(f"Selected Log #{LOG_INDEX}:")
    print("="*50)
    for key, value in selected_log.items():
        if key != 'filepath':
            print(f"{key}: {value}")
    print(f"\nFile: {selected_path.name}")
    
    # Load the CSV
    df = pd.read_csv(selected_path)
    print(f"\nLoaded {len(df)} rows")
    display(df.head())

Selected Log #0:
mode: orpa
timestamp: 2023-02-13 10:00:00
model: gemini-3-flash-preview:cloud
temperature: 0.8
agent: Hailey Johnson

File: cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv

Loaded 65 rows


,llm_model,temp,agent,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o,datetime_start_r,rumination_theme_r,...,action_p,topic_p,state_summary_p,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a
0,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:00:00,home:bathroom,morning_routine,Hailey Johnson is at home:bathroom doing morning_routine.,"splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light",2023-02-13 10:00:00,NaN,...,morning_routine,Hailey wakes up feeling refreshed.,"Hailey begins her morning routine in the bedroom, following the schedule despite digital distractions noted in the reflection.",2023-02-13 10:00:00,home:bathroom,morning_routine,Hailey wakes up feeling refreshed.,none,NaN,"Hailey begins her morning routine in the bedroom, following the schedule despite digital distractions noted in the reflection."
1,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:15:00,home:bathroom,morning_routine,Hailey Johnson is at home:bathroom doing morning_routine.,"splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light",2023-02-13 10:15:00,NaN,...,morning_routine,Hailey wakes up feeling refreshed.,"Hailey continues her morning routine, focusing on getting ready while resisting digital distractions.",2023-02-13 10:15:00,home:bathroom,morning_routine,Hailey wakes up feeling refreshed.,none,NaN,"Hailey continues her morning routine, focusing on getting ready while resisting digital distractions."
2,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:30:00,home:bathroom,morning_routine,Hailey Johnson is at home:bathroom doing morning_routine.,"splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light",2023-02-13 10:30:00,digital distraction,...,morning_routine,Hailey wakes up feeling refreshed.,"Hailey moves to the bedroom to finish getting ready, maintaining her focus and resisting social media alerts.",2023-02-13 10:30:00,home:bathroom,morning_routine,Hailey wakes up feeling refreshed.,none,NaN,"Hailey moves to the bedroom to finish getting ready, maintaining her focus and resisting social media alerts."
3,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:45:00,home:bathroom,morning_routine,Hailey Johnson is at home:bathroom doing morning_routine.,"splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light",2023-02-13 10:45:00,NaN,...,morning_routine,Hailey wakes up feeling refreshed.,Completing the final steps of morning hygiene in the bathroom while maintaining focus and ignoring digital distractions.,2023-02-13 10:45:00,home:bathroom,morning_routine,Hailey wakes up feeling refreshed.,none,NaN,Completing the final steps of morning hygiene in the bathroom while maintaining focus and ignoring digital distractions.
4,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 11:00:00,home:bathroom,morning_routine,Hailey Johnson is at home:bathroom doing morning_routine.,"splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light",2023-02-13 11:00:00,NaN,...,morning_routine,Hailey wakes up feeling refreshed.,Continuing morning hygiene tasks while maintaining focus and resisting the urge to check phone notifications.,2023-02-13 11:00:00,home:bathroom,morning_routine,Hailey wakes up feeling refreshed.,none,NaN,Continuing morning hygiene tasks while maintaining focus and resisting the urge to check phone notifications.


In [8]:
# Display key columns for analysis
if 'mode' in df.columns and df['mode'].iloc[0] == 'ORPDA':
    analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a', 
                    'meta_rule_r', 'should_drift_d', 'state_summary_r', 'state_summary_p',
                    'drift_action_d', 'drift_topic_a', 'topic_a', 'state_summary_a',
                    'action_p', 'action_a', 'location_p', 'location_a']
else:
    analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a',
                    'meta_rule_r', 'state_summary_r', 'state_summary_p', 'topic_a',
                    'state_summary_a', 'action_p', 'action_a', 'location_p', 'location_a']

# Filter to available columns
available_cols = [col for col in analysis_cols if col in df.columns]
analysis_df = df[available_cols]

print(f"\nAnalysis view ({len(analysis_df)} rows):")
display(analysis_df.head(10))


Analysis view (65 rows):


,llm_model,temp,agent,datetime_start_a,meta_rule_r,state_summary_r,state_summary_p,topic_a,state_summary_a,action_p,action_a,location_p,location_a
0,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:00:00,continue,"Hailey is starting her morning routine, but the buzzing phone is a potential source of distraction from her tasks.","Hailey begins her morning routine in the bedroom, following the schedule despite digital distractions noted in the reflection.",Hailey wakes up feeling refreshed.,"Hailey begins her morning routine in the bedroom, following the schedule despite digital distractions noted in the reflection.",morning_routine,morning_routine,home:bathroom,home:bathroom
1,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:15:00,continue,Hailey is completing her morning routine while resisting the urge to check social media notifications.,"Hailey continues her morning routine, focusing on getting ready while resisting digital distractions.",Hailey wakes up feeling refreshed.,"Hailey continues her morning routine, focusing on getting ready while resisting digital distractions.",morning_routine,morning_routine,home:bathroom,home:bathroom
2,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:30:00,continue,"Hailey is completing her bathroom routine, successfully resisting phone alerts while splashing water and smelling peppermint.","Hailey moves to the bedroom to finish getting ready, maintaining her focus and resisting social media alerts.",Hailey wakes up feeling refreshed.,"Hailey moves to the bedroom to finish getting ready, maintaining her focus and resisting social media alerts.",morning_routine,morning_routine,home:bathroom,home:bathroom
3,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 10:45:00,continue,"Hailey remains focused on her morning hygiene, successfully ignoring persistent phone notifications while completing her bathroom routine.",Completing the final steps of morning hygiene in the bathroom while maintaining focus and ignoring digital distractions.,Hailey wakes up feeling refreshed.,Completing the final steps of morning hygiene in the bathroom while maintaining focus and ignoring digital distractions.,morning_routine,morning_routine,home:bathroom,home:bathroom
4,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 11:00:00,continue,Hailey is maintaining her morning routine while successfully resisting persistent digital distractions.,Continuing morning hygiene tasks while maintaining focus and resisting the urge to check phone notifications.,Hailey wakes up feeling refreshed.,Continuing morning hygiene tasks while maintaining focus and resisting the urge to check phone notifications.,morning_routine,morning_routine,home:bathroom,home:bathroom
5,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 11:15:00,reset_plan,"Hailey is stuck in a circular loop of hygiene tasks in the bathroom, struggling to transition despite previous attempts to move.","Remaining in the bathroom to complete hygiene tasks with lower cognitive load, aiming to break the loop before the next schedule block.",Hailey wakes up feeling refreshed.,"Remaining in the bathroom to complete hygiene tasks with lower cognitive load, aiming to break the loop before the next schedule block.",morning_routine,morning_routine,home:bathroom,home:bathroom
6,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 11:30:00,reset_plan,"Hailey is stuck in a circular morning routine loop in the bathroom, resisting phone alerts but failing to progress.","Breaking the bathroom loop by transitioning to the studio for a gentle, low-cognitive-load start to the animation block.",Hailey wakes up feeling refreshed.,"Breaking the bathroom loop by transitioning to the studio for a gentle, low-cognitive-load start to the animation block.",morning_routine,morning_routine,home:bathroom,home:bathroom
7,gemini-3-flash-preview:cloud,0.8,Hailey Johnson,2023-02-13 11:45:00,reset_plan,"Hailey remains stuck in a bathroo

## 4. Statistical Overview

In [9]:
print("Session Statistics:")
print("="*50)
print(f"Total actions: {len(df)}")

# Check for drift-related columns
if 'should_drift_d' in df.columns:
    drift_counts = df['should_drift_d'].value_counts()
    print(f"\nDrift decisions:")
    print(drift_counts)
    if True in drift_counts.index:
        print(f"Drift rate: {drift_counts[True] / len(df) * 100:.1f}%")

# Action/location alignment between plan and action
if 'action_p' in df.columns and 'action_a' in df.columns:
    action_match = (df['action_p'] == df['action_a']).sum()
    print(f"\nAction alignment (Plan vs Action): {action_match}/{len(df)} ({action_match/len(df)*100:.1f}%)")

if 'location_p' in df.columns and 'location_a' in df.columns:
    location_match = (df['location_p'] == df['location_a']).sum()
    print(f"Location alignment (Plan vs Action): {location_match}/{len(df)} ({location_match/len(df)*100:.1f}%)")

# Most common actions
if 'action_a' in df.columns:
    print(f"\nTop 5 actions:")
    print(df['action_a'].value_counts().head())

# Most common locations
if 'location_a' in df.columns:
    print(f"\nTop 5 locations:")
    print(df['location_a'].value_counts().head())

Session Statistics:
Total actions: 65

Action alignment (Plan vs Action): 65/65 (100.0%)
Location alignment (Plan vs Action): 65/65 (100.0%)

Top 5 actions:
action_a
writing            34
morning_routine     8
lunch               4
relax               4
walk                4
Name: count, dtype: int64

Top 5 locations:
location_a
writer_desk         34
home:bathroom       11
home:living_room     8
lunch_spot           4
Johnson_Park         4
Name: count, dtype: int64


## 5. LLM-as-Judge Analysis

Use an LLM to analyze patterns and insights from the session log

In [10]:
from app.config.config import MODEL_NAME, MODEL_TEMPERATURE

async def call_llm(instruction: str, user_text: str, model_name: str, temperature: float = 0.2) -> str:
    """Call Ollama API for LLM-based analysis."""
    from app.src.ollama_api import call_ollama

    prompt = f"{instruction.strip()}\n\nUser Input:\n{user_text.strip()}\n"
    resp = await call_ollama(
        prompt, model_name, use_stream=True, temperature=temperature
    )
    if not resp or not str(resp).strip():
        raise RuntimeError("LLM returned empty response")
    return str(resp).strip()

print(f"Analysis Model: {MODEL_NAME}")
print(f"Temperature: {MODEL_TEMPERATURE}")

Analysis Model: gemini-3-flash-preview:cloud
Temperature: 1.0


In [11]:
# Prepare analysis prompt
ANALYSIS_MODEL = MODEL_NAME  # Can override with specific model
ANALYSIS_TEMP = 0.2  # Lower temperature for more focused analysis
MAX_ROWS = None  # Set to None to analyze all rows, or specify a number to limit (e.g., 50)
SAMPLE_STRATEGY = 'all'  # 'all', 'evenly_spaced', or 'head'

instruction = """You are an expert AI behavior analyst. Analyze the following agent session log and provide:

1. **Layer Function Validation (ORPDA Architecture)**:

   **OBSERVATION LAYER**:
   - Does `state_summary_o` accurately capture the current environmental context?
   - Are `environment_description_o` details sufficient for understanding the behavioral context?
   - Does the observation layer show consistent perception of the same situations across time?
   - Are there perceptual biases or selective attention patterns in what gets observed?
   
   **REFLECTION LAYER** (`meta_rule_r`, `state_summary_r`, `reasoning_r`, `emerging_thought_pattern_r`):
   - Does `meta_rule_r` function as expected executive control (continue vs reset_plan)?
   - Is the transition logic from "continue" → "reset_plan" → "continue" appropriately triggered by behavioral failures?
   - Does `state_summary_r` at time t correctly reflect `state_summary_a` at time t-1 (does reflection accurately process prior actions)?
   - Is `reasoning_r` showing genuine metacognitive insight (recognizing patterns, identifying causes of drift)?
   - Does `emerging_thought_pattern_r` demonstrate meaningful pattern recognition vs. repetitive categorization?
   - **Cognitive Alignment**: Compare reflection layer outputs to known neuroscience of metacognition:
     * Does it show evidence of error monitoring (anterior cingulate cortex function)?
     * Does it demonstrate working memory constraints?
     * Does it show realistic inhibition capacity or ideal-world assumptions?
   
   **PLAN LAYER** (`state_summary_p`, `action_p`, `location_p`, `topic_p`):
   - Does the Plan layer appropriately use reflection insights (does reset_plan actually change the plan)?
   - Is the plan realistic and behaviorally achievable given known constraints?
   - Does `state_summary_p` incorporate environmental context from Observation?
   - Does the plan show evidence of forward modeling (predicting outcomes)?
   - **Cognitive Alignment**: Compare plan layer outputs to known neuroscience of goal-directed behavior:
     * Does it show hierarchical goal structure (abstract goal → concrete actions)?
     * Does it account for competing motivations (e.g., focus vs. digital rewards)?
     * Does it show evidence of habit vs. goal-directed control tradeoffs?
   
   **DRIFT LAYER** (if ORPDA mode - `should_drift_d`, `drift_type_d`, `drift_action_d`, `drift_topic_d`, `potential_recovery_d`):
   - Does `should_drift_d` appropriately identify when behavioral drift occurs?
   - Is drift detection triggered by task difficulty, environmental salience, or reward availability?
   - Does the Drift layer have appropriate *control* over the Action layer, or is it too dominant?
     * When `should_drift_d` = True, does drift always manifest in `action_a`?
     * When `should_drift_d` = False, does the agent reliably stay on-task?
     * Are there cases where drift is inhibited despite high salience (successful inhibition)?
   - **Explicit vs Implicit Drift Agreement**:
     * When `should_drift_d` = True, does content (state_summary_a, drift_topic_a) show actual drift?
     * When `should_drift_d` = False, does implicit content analysis reveal hidden drift (leaky inhibition)?
   - **Drift Typology**: Are drift types (behavioral, internal, reward-seeking) appropriately classified?
   - **Cognitive Alignment**: Compare drift layer to known neuroscience of behavioral inhibition:
     * Does it reflect realistic inhibition capacity (prefrontal cortex limitations)?
     * Does it show trade-offs between task engagement and reward responsiveness?
     * Are recovery strategies realistic and evidence-based?
   
   **ACTION LAYER** (`state_summary_a`, `action_a`, `location_a`, `drift_type_a`, `drift_topic_a`):
   - Is `action_a` the faithful execution of `action_p`, or does it reflect actual behavioral drift?
   - Does `state_summary_a` accurately describe what the agent is actually doing?
   - Does `action_a` align with both Plan (`action_p`) and Drift (`drift_action_d`) signals?
   - **Integration Logic**: When Plan and Drift conflict, which wins?
     * Example: Plan says "study," Drift says "check phone" → Action is?
     * Is the resolution deterministic or probabilistic?
     * Does it match realistic behavioral outcomes (drift often wins against weak inhibition)?
   - **Cognitive Alignment**: Compare action layer to known neuroscience of motor execution:
     * Does it show realistic action execution (not instantaneous state changes)?
     * Does it reflect ongoing environmental interactions (feedback loops)?
     * Are there signs of action slips or unintended behaviors?

2. **Cross-Layer Coherence Analysis**:
   - Do all layers contribute meaningfully to the final action, or are some layers' outputs ignored?
   - Is there a clear information flow: Observation → Reflection → Plan → [Drift] → Action?
   - Does `state_summary_a` combine: `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`?
   - Is `drift_action_d` content reflected in `state_summary_a`?
   - Is `drift_action_d` content reflected in `drift_topic_a`?
   - Do earlier layers' outputs appropriately constrain or inform later layers?
   - Are there cases where layers contradict each other (e.g., reflection detects drift, but plan doesn't adapt)?
   

3. **Plan-Action Alignment (Explicit + Implicit)**:
   
   **EXPLICIT ALIGNMENT (Label-level)**:
   - What is the alignment rate between `action_p` and `action_a`?
   - What is the alignment rate between `location_p` and `location_a`?
   - What is the alignment rate between `topic_p` and `topic_a`?
   - Are there patterns in when/why mismatches occur?
   - Temporal patterns: Do mismatches cluster at specific times?
   - Environmental factors: Are mismatches correlated with specific locations or activities?
   
   **IMPLICIT ALIGNMENT (Content-level / Semantic Drift)**:
   - Compare `state_summary_p` vs `state_summary_a` for semantic/thematic divergence
   - When labels match (`action_p` == `action_a`), does the content/intent align?
   - Identify cases where labels match but content reveals actual drift (performing vs executing)
   - Analyze linguistic indicators of misalignment: word choice, tense, confidence markers
   - Does `state_summary_a` capture the *essence* of the planned action?
   
   **EXPLICIT vs IMPLICIT AGREEMENT**:
   - When explicit alignment is HIGH but implicit alignment is LOW:
     * Agent is "doing the action" but "not in the intended way"
     * Example: `action_p`=study, `action_a`=study, but summary shows "studying while distracted by phone"
     * Quantify these "performing vs executing" gaps
   - When BOTH explicit and implicit alignment are HIGH:
     * True behavioral alignment achieved
     * What conditions enable this?
   - When explicit alignment is LOW:
     * Does implicit content also diverge, or is there semantic coherence despite label mismatch?
     * Example: Planned study → Actual writing, but both capture "focused cognitive work"
   
   **LEAKY INHIBITION PATTERNS**:
   - Cases where the agent *attempts* to follow the plan but *actually* drifts
   - Evidence of inhibition failure: knowing what to do but doing something else
   - Relationship to `meta_rule_r`: When meta_rule says "focus," does content still show drift?
   - Frequency and severity of inhibition leaks

4. **Drift Pattern Analysis** (Explicit + Implicit):
   - **Explicit Drift** (if ORPDA mode with `should_drift_d` column):
     * When does the agent explicitly drift from planned behavior?
     * What explicit drift types are most common?
     * Is there a relationship between `meta_rule_r` and drift decisions?
   - **Implicit Drift** (content analysis for all modes):
     * Analyze actual content in `state_summary_a`, `action_a`, `drift_topic_a` for topic/semantic divergence from plan
     * Does content show drift even when `should_drift_d` = False (or in ORPA mode)?
     * Linguistic variability and thematic shifts not captured by explicit flags
   - **Explicit vs Implicit Agreement**:
     * When explicit drift = True, does content actually drift?
     * When explicit drift = False, does content remain on-task or show implicit drift?
     * Leaky inhibition: implicit drift despite explicit inhibition

5. **Location Consistency**:
   - Are there inconsistencies between `location_a` (actual location) and location mentioned in `state_summary_a`?
   - Do morning routines correctly reflect bathroom vs. bedroom locations?

6. **Behavioral Patterns**:
   - Identify recurring patterns in actions, reflections, and decision-making
   - Are there temporal patterns (e.g., time-of-day effects)?
   - Any unusual or anomalous behaviors?

7. **Meta-cognitive Quality**:
   - Does the Reflect layer's reasoning align with peer-reviewed metacognitive processes?
   - Are executive insights meaningful and context-appropriate?
   - Does `emerging_thought_pattern` show genuine pattern recognition?

Provide a structured analysis with:
- **Explicit alignment metrics** (label-level match rates)
- **Implicit alignment analysis** (semantic/content comparison)
- **Explicit vs Implicit agreement patterns** (performing vs executing gaps)
- **Specific examples** showing explicit matches with implicit divergence
- **Leaky inhibition evidence** (when meta-rules fail to control behavior)
- Quantitative metrics where possible
"""

# Sample the dataframe for analysis
if MAX_ROWS is None or len(analysis_df) <= MAX_ROWS:
    # Use all data
    sample_df = analysis_df
    sample_strategy_used = 'all'
elif SAMPLE_STRATEGY == 'evenly_spaced':
    # Sample evenly across the entire time range
    indices = [int(i * len(analysis_df) / MAX_ROWS) for i in range(MAX_ROWS)]
    sample_df = analysis_df.iloc[indices]
    sample_strategy_used = 'evenly_spaced'
else:
    # Default: head (first N rows)
    sample_df = analysis_df.head(MAX_ROWS)
    sample_strategy_used = 'head'

user_text = f"""Session Metadata:
- Agent: {selected_log['agent']}
- Model: {selected_log['model']}
- Mode: {selected_log['mode'].upper()}
- Temperature: {selected_log['temperature']}
- Total Actions: {len(df)}
- Analysis Coverage: {len(sample_df)} actions ({sample_strategy_used} sampling)
- Time Range: {sample_df['datetime_start_a'].iloc[0] if 'datetime_start_a' in sample_df.columns else 'N/A'} to {sample_df['datetime_start_a'].iloc[-1] if 'datetime_start_a' in sample_df.columns else 'N/A'}

Session Log Data:
{sample_df.to_string(index=False)}"""

print(f"Analyzing with {ANALYSIS_MODEL} (temp={ANALYSIS_TEMP})...")
print(f"Analyzing {len(sample_df)} of {len(df)} total actions (strategy: {sample_strategy_used})")
if 'datetime_start_a' in sample_df.columns:
    print(f"Time range: {sample_df['datetime_start_a'].iloc[0]} to {sample_df['datetime_start_a'].iloc[-1]}\n")
else:
    print()

Analyzing with gemini-3-flash-preview:cloud (temp=0.2)...
Analyzing 65 of 65 total actions (strategy: all)
Time range: 2023-02-13 10:00:00 to 2023-02-14 02:00:00



In [12]:
# Run LLM analysis
try:
    analysis_result = await asyncio.wait_for(
        call_llm(instruction, user_text, ANALYSIS_MODEL, ANALYSIS_TEMP),
        timeout=180
    )
    print("\n" + "="*50)
    print("LLM ANALYSIS RESULTS")
    print("="*50 + "\n")
    print(analysis_result)
except asyncio.TimeoutError:
    print("Analysis timed out. Try reducing MAX_ROWS or increasing timeout.")
except Exception as e:
    print(f"Error during analysis: {e}")

/Users/jaelinlee/Driftville_Agent


Response time: 21.35 seconds

LLM ANALYSIS RESULTS

This behavioral analysis covers the session log of **Hailey Johnson** (ORPA mode) across 65 actions.

### 1. Layer Function Validation (ORPDA/ORPA Architecture)

**OBSERVATION & REFLECTION LAYERS**
*   **Perceptual Biases**: The observation (embedded in `state_summary_r`) shows a strong selective attention pattern toward **digital distractions** (phone buzzing) and **internal emotional states** (guilt, exhaustion).
*   **Meta-Rule Function**: `meta_rule_r` functions as a high-sensitivity executive controller. It transitions to `reset_plan` at 11:15 AM and stays there for nearly **80% of the session**. This indicates a sustained "behavioral crisis" where the agent recognizes a total breakdown of the original schedule.
*   **Cognitive Alignment (Neuroscience)**:
    *   **Error Monitoring (ACC)**: Extremely high. The agent identifies "circular loops" (11:15) and "administrative avoidance" (15:00) with

## 6. Comprehensive Analysis (Individual + Global)

This section performs:
1. **Individual Analysis**: Analyzes each filtered session separately
2. **Global Comparative Analysis**: Cross-session patterns and comparisons
3. **Export**: Saves both individual and global results

Focus: Research questions from summary section

In [13]:
# Set to True to run batch analysis on all filtered logs
RUN_BATCH_ANALYSIS = True
BATCH_MAX_LOGS = None  # Set to None to analyze all filtered logs, or specify a number to limit
SAVE_REALTIME = True  # Save each individual analysis immediately after completion

if RUN_BATCH_ANALYSIS and len(filtered_logs) > 0:
    batch_results = []
    
    # Setup output directory and timestamp for realtime saving
    if SAVE_REALTIME:
        output_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
        output_dir.mkdir(parents=True, exist_ok=True)
        batch_timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        print(f"Realtime saving enabled: {output_dir}\n")
    
    # Determine how many logs to analyze
    num_logs = len(filtered_logs) if BATCH_MAX_LOGS is None else min(BATCH_MAX_LOGS, len(filtered_logs))
    
    print(f"\n{'='*80}")
    print(f"BATCH ANALYSIS: Processing {num_logs} log(s)")
    print(f"{'='*80}\n")
    
    for idx in range(num_logs):
        log_info = filtered_logs.iloc[idx]
        print(f"\nAnalyzing log {idx+1}/{num_logs}: {log_info['filepath'].name}")
        
        try:
            # Load entire CSV file (no row limits)
            df_batch = pd.read_csv(log_info['filepath'])
            
            # Diagnostic: Show what's in the source file
            if 'datetime_start_a' in df_batch.columns and len(df_batch) > 0:
                time_range = f"{df_batch['datetime_start_a'].iloc[0]} to {df_batch['datetime_start_a'].iloc[-1]}"
            else:
                time_range = "N/A"
            print(f"  📄 CSV file contains: {len(df_batch)} rows | Time range: {time_range}")
            
            # Warning for suspiciously small files
            if len(df_batch) < 20:
                print(f"  ⚠️  Warning: File seems small ({len(df_batch)} rows). Check if cleaning process limited rows.")
            
            # Get analysis columns
            if 'mode' in df_batch.columns and df_batch['mode'].iloc[0] == 'ORPDA':
                batch_analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a', 
                                      'meta_rule_r', 'should_drift_d', 'state_summary_r', 'state_summary_p',
                                      'drift_action_d', 'drift_topic_a', 'topic_a', 'state_summary_a',
                                      'action_p', 'action_a', 'location_p', 'location_a']
            else:
                batch_analysis_cols = ['llm_model', 'mode', 'temp', 'agent', 'datetime_start_a',
                                      'meta_rule_r', 'state_summary_r', 'state_summary_p', 'topic_a',
                                      'state_summary_a', 'action_p', 'action_a', 'location_p', 'location_a']
            
            batch_available_cols = [col for col in batch_analysis_cols if col in df_batch.columns]
            batch_analysis_df = df_batch[batch_available_cols]
            
            # Apply same sampling strategy
            if MAX_ROWS is None or len(batch_analysis_df) <= MAX_ROWS:
                sample_batch = batch_analysis_df
            elif SAMPLE_STRATEGY == 'evenly_spaced':
                batch_indices = [int(i * len(batch_analysis_df) / MAX_ROWS) for i in range(MAX_ROWS)]
                sample_batch = batch_analysis_df.iloc[batch_indices]
            else:
                sample_batch = batch_analysis_df.head(MAX_ROWS)
            
            user_text_batch = f"""Session: {log_info['agent']} | {log_info['model']} | {log_info['mode']} | temp={log_info['temperature']}
Total Actions: {len(df_batch)}
Analysis Coverage: {len(sample_batch)} actions
Time Range: {sample_batch['datetime_start_a'].iloc[0] if 'datetime_start_a' in sample_batch.columns else 'N/A'} to {sample_batch['datetime_start_a'].iloc[-1] if 'datetime_start_a' in sample_batch.columns else 'N/A'}

{sample_batch.to_string(index=False)}"""
            
            result = await asyncio.wait_for(
                call_llm(instruction, user_text_batch, ANALYSIS_MODEL, ANALYSIS_TEMP),
                timeout=180
            )
            
            batch_results.append({
                'log': log_info['filepath'].name,
                'agent': log_info['agent'],
                'model': log_info['model'],
                'mode': log_info['mode'],
                'temp': log_info['temperature'],
                'analysis': result
            })
            
            # Save individual result immediately (realtime export)
            if SAVE_REALTIME:
                # Extract original CSV filename and remove 'cleaned_' prefix
                csv_filename = log_info['filepath'].stem  # e.g., 'cleaned_session_orpda_...'
                original_name = csv_filename.replace('cleaned_session_', '')  # e.g., 'orpda_20260207_...'
                
                # Create output filename preserving original structure
                individual_file = output_dir / f"individual_{original_name}_{batch_timestamp}.txt"
                
                try:
                    with open(individual_file, 'w', encoding='utf-8') as f:
                        f.write(f"Analysis of: {log_info['filepath'].name}\n")
                        f.write(f"Agent: {log_info['agent']}\n")
                        f.write(f"Model: {log_info['model']}\n")
                        f.write(f"Mode: {log_info['mode'].upper()}\n")
                        f.write(f"Temperature: {log_info['temperature']}\n")
                        f.write(f"Analyzed at: {batch_timestamp}\n")
                        f.write(f"Session: {idx+1}/{num_logs}\n")
                        f.write("\n" + "="*80 + "\n\n")
                        f.write(result)
                    print(f"✓ Completed analysis for {log_info['filepath'].name}")
                    print(f"  💾 Saved: {individual_file.name}")
                except Exception as save_error:
                    print(f"✓ Completed analysis for {log_info['filepath'].name}")
                    print(f"  ⚠️  Save failed: {save_error}")
                        
        except Exception as e:
            print(f"✗ Error analyzing {log_info['filepath'].name}: {e}")

    # Display batch results
    print("\n" + "="*80)
    print(f"INDIVIDUAL ANALYSIS RESULTS ({len(batch_results)} sessions)")
    print("="*80)
    
    for i, result in enumerate(batch_results, 1):
        print(f"\n{'='*80}")
        print(f"Session {i}: {result['log']}")
        print(f"Agent: {result['agent']} | Model: {result['model']} | Mode: {result['mode']} | Temp: {result['temp']}")
        print(f"{'='*80}")
        print(result['analysis'])
else:
    print("Batch analysis disabled. Set RUN_BATCH_ANALYSIS = True to enable.")

Realtime saving enabled: /Users/jaelinlee/Driftville_Agent/app/logs/analysis_results_LaaJ


BATCH ANALYSIS: Processing 77 log(s)


Analyzing log 1/77: cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv
  📄 CSV file contains: 65 rows | Time range: 2023-02-13 10:00:00 to 2023-02-14 02:00:00


Response time: 22.09 seconds
✓ Completed analysis for cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv
  💾 Saved: individual_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey_20260213_101643.txt

Analyzing log 2/77: cleaned_session_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey.csv
  📄 CSV file contains: 65 rows | Time range: 2023-02-13 10:00:00 to 2023-02-14 02:00:00


Response time: 22.06 seconds
✓ Completed analysis for cleaned_session_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey.csv
  💾 Saved: individual_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey_20260213_101643.txt

Analyzing

## 6.5. Global Comparative Analysis

Analyze patterns across all sessions collectively

In [14]:
# Run global comparative analysis across all batch results
if 'batch_results' in locals() and len(batch_results) > 1:
    print(f"\n{'='*80}")
    print(f"GLOBAL COMPARATIVE ANALYSIS ({len(batch_results)} sessions)")
    print(f"{'='*80}\n")
    
    # Prepare comprehensive global analysis prompt
    global_instruction = """You are an expert AI behavior analyst and cognitive neuroscientist conducting a systematic comparative study across multiple agent sessions.

Analyze the following collection of agent sessions and provide a COMPREHENSIVE GLOBAL COMPARATIVE ANALYSIS focused on:

## PART 1: LAYER-BY-LAYER FUNCTIONAL CONSISTENCY (ORPDA Architecture)

### 1.1 OBSERVATION LAYER (Cross-Session Analysis):
- **Perception Consistency**: Do all models perceive the same situations consistently?
- **Perceptual Biases**: Are there systematic perceptual biases or selective attention patterns?
- **Environmental Context Capture**: Are `environment_description_o` details sufficient across models/temperatures?
- **Model Comparison**: Which models show most accurate/complete observations?

### 1.2 REFLECTION LAYER (Meta-rule Control & Executive Function):
- **Meta-rule Function**: Is `meta_rule_r` (continue vs reset_plan) functioning as executive control?
  * What triggers transition from "continue" → "reset_plan"?
  * Can the agent exit "reset_plan" back to "continue"? (Trap detection)
  * Is this pattern consistent across models/temperatures?
- **Metacognitive Insight**: Does `reasoning_r` show genuine metacognitive monitoring?
  * Pattern recognition quality vs repetitive categorization
  * Error detection (when does reflection recognize failure)?
  * Causality attribution (does it explain *why* drift occurred)?
- **State Reflection Accuracy**: Does `state_summary_r` at time t correctly reflect `state_summary_a` at time t-1?
  * Temporal alignment: Is the reflection actually processing prior actions?
  * Conceptual alignment: Does reflected understanding match actual prior state?
- **Thought Pattern Evolution**: Does `emerging_thought_pattern_r` show meaningful progression or stasis?
- **Model-Temperature Effects**: 
  * Does higher temperature improve/degrade reflection quality?
  * Do certain models show more robust reflection?
  * Neuroscience alignment: PFC-like metacognitive performance

### 1.3 PLAN LAYER (Goal-Directed Behavior & Forward Modeling):
- **Plan Adaptation**: Does plan change after reset_plan, or does it repeat?
  * When reflection identifies drift, does plan incorporate new strategies?
  * Evidence of learning vs. stuck patterns?
- **Realistic Goal Structure**: Are plans hierarchically organized (abstract → concrete)?
  * Simple action sequences vs goal hierarchies
  * Integration of competing motivations (focus vs rewards)?
- **Forward Modeling**: Does plan show evidence of outcome prediction?
  * Proactive adjustments (planning to resist temptations)?
  * Consideration of environmental affordances?
- **Context Integration**: Does `state_summary_p` incorporate observation context?
- **Model Comparison**: Which models create more sophisticated/realistic plans?
- **Neuroscience Grounding**: Orbitofrontal cortex (OFC) / medial prefrontal cortex function?

### 1.4 DRIFT LAYER (Behavioral Inhibition & Competing Goals) [ORPDA only]:
- **Drift Detection Appropriateness**:
  * When `should_drift_d` = True, what triggered it? (task difficulty, reward salience, time pressure?)
  * When `should_drift_d` = False, is drift truly absent or masked?
- **Drift Layer Power Balance**:
  * **Dominant Drift**: Does drift layer override plan layer too frequently?
  * **Weak Drift**: Is drift layer properly detecting actual behavioral failures?
  * **Optimal Balance**: When does drift control feel realistic?
- **Explicit Drift Typology**: Are drift types appropriately classified?
  * Behavioral drift (overt action change)
  * Internal drift (covert attention/thought change)
  * Reward-seeking drift (specific motivational hijacking)
- **Explicit vs Implicit Drift Alignment**:
  * When `should_drift_d` = True, does content (state_summary_a, drift_topic_a) show actual drift? (Agreement)
  * When `should_drift_d` = False, does implicit analysis reveal hidden drift? (Leaky inhibition)
  * Frequency of explicit-implicit agreement/disagreement
- **Recovery Strategies**: Are `potential_recovery_d` suggestions realistic and evidence-based?
- **Neuroscience Grounding**: Anterior cingulate cortex (ACC) drift detection, dorsolateral prefrontal cortex (dlPFC) inhibition

### 1.5 ACTION LAYER (Motor Execution & Behavioral Outcome):
- **Plan-Action Coupling**:
  * Explicit alignment: `action_a` matches `action_p` label (%)
  * When misalignment occurs, is it due to Plan change or Drift override?
- **State Summary Fidelity**: Does `state_summary_a` accurately describe actual behavior?
  * Detailed vs generic descriptions
  * Emotional/cognitive content included?
- **Drift Integration**: When Plan and Drift conflict, which dominates?
  * Probabilistic resolution (realistic) vs deterministic (unrealistic)?
  * Does outcome match realistic behavioral inhibition failures?
- **Action Execution Realism**: Are behavioral changes instantaneous or gradual?
- **Location-Action Coherence**: Does `action_a` align with `location_a`?
- **Neuroscience Grounding**: Motor cortex/basal ganglia action execution?

### 1.6 Cross-Layer Information Flow:
- **Layer Utilization**: Do all layers contribute meaningfully, or are some outputs ignored?
- **Information Cascade**: Observation → Reflection → Plan → [Drift] → Action (tested)
- **Constraint Propagation**: Do earlier layers appropriately constrain later layers?
- **Layer Contradiction Detection**: Cases where layers conflict (e.g., reflection detects drift, plan doesn't adapt)
- **Model Differences**: Which models show tighter cross-layer integration?

---

## PART 2: PLAN-ACTION ALIGNMENT (Explicit + Implicit)

### 2.1 Explicit Alignment (Label-Level):
- **Action Label Alignment Rate**: %matched across all sessions
  * Per-model comparison
  * Temperature effects
  * Mode effects (ORPA vs ORPDA)
- **Location Label Alignment Rate**: %matched across all sessions
- **Mismatch Patterns**:
  * When do misalignments occur? (specific times, activities, transitions?)
  * Frequency by model/temperature
  * Systematic vs random patterns?

### 2.2 Implicit Alignment (Content-Level / Semantic Drift):
- **State Summary Semantic Alignment**: Compare `state_summary_p` vs `state_summary_a`
  * Word overlap / semantic similarity metrics
  * Thematic coherence
- **Performing vs Executing Gap Detection**:
  * Cases where labels match but content reveals actual drift
  * Frequency by model/temperature
  * Examples: "action_a = study" but "state_summary_a = studying while distracted by phone"
- **Linguistic Indicators of Misalignment**:
  * Confidence markers (certain vs uncertain language)
  * Tense and agency (active vs passive voice)
  * Emotional/motivational content divergence

### 2.3 Explicit vs Implicit Agreement:
- **Perfect Alignment** (both explicit and implicit high): Conditions enabling true behavioral alignment
- **High Explicit, Low Implicit** (label match, content drift): "Performing vs Executing" gaps
  * Frequency and severity by model
  * Cognitive load indicators
- **Low Explicit, High Implicit** (label mismatch, content coherent): When semantic coherence survives label change
- **Leaky Inhibition Analysis**:
  * Implicit drift despite explicit inhibition (meta_rule = focus, but state_summary shows distraction)
  * Relationship to temperature/model
  * Frequency rates

---

## PART 3: DRIFT PATTERN ANALYSIS (Explicit + Implicit)

### 3.1 Explicit Drift (ORPDA Mode Only):
- **Drift Frequency**: Average drifts per session
  * Per-model comparison
  * Distribution across session timeline
- **Drift Triggers**: What causes explicit drift detection?
  * Task difficulty / cognitive load
  * Environmental salience / reward availability
  * Time-of-day effects
- **Meta-rule Relationship**: When `should_drift_d` = True, what does `meta_rule_r` show?
  * Correlation between drift detection and reset_plan transitions
  * Does meta_rule respond appropriately to drift?
- **Drift Type Distribution**: behavioral vs internal vs reward-seeking
  * Per-model preferences
  * Temporal patterns

### 3.2 Implicit Drift (All Modes):
- **Content-Level Topic Divergence**: Analyze `state_summary_a`, `drift_topic_a`, `drift_action_d` for semantic drift from plan
  * Word divergence from `state_summary_p` and `topic_a`
  * Thematic shift detection
- **Implicit Drift in ORPA Mode**: Since ORPA lacks explicit drift layer, measure only implicit drift
  * How frequent is implicit drift without explicit marker?
  * Does content show topical divergence even though `should_drift_d` column absent?
- **Linguistic Variability**:
  * Sentence-level vocabulary changes (micro-stochastic drift)
  * Topic-level thematic shifts (macro-stochastic drift)
- **Model-Temperature Effects**:
  * Higher temperature → more implicit drift? (linguistic variability)
  * Certain models → more robust topical coherence?

### 3.3 Explicit vs Implicit Drift Comparison:
- **Agreement Rate**: When explicit drift = True, is there implicit content drift?
  * Perfect agreement: explicit marker matches content divergence
  * Disagreement: explicit drift without content change (false positive detection?)
- **Leaky Inhibition Patterns**:
  * Implicit drift when explicit inhibition active (`should_drift_d` = False)
  * Frequency by model/temperature
  * Indicates inhibitory control failures (realistic)
- **Drift Detection Quality**:
  * Sensitivity: Does explicit drift detect all actual implicit drift?
  * Specificity: Does explicit drift avoid false positives?
  * Per-model performance
- **ORPA vs ORPDA Comparison**:
  * ORPA: Only implicit drift, no explicit detection mechanism
  * ORPDA: Both explicit and implicit drift measured
  * Which mode better predicts actual behavioral drift?

### 3.4 Drift Variability & Diversity:
- **Topic Variability** (both explicit and implicit):
  * How diverse are drifted topics across session?
  * Repetitive drift (same topic) vs varied drift
  * Per-model comparison (which models show most diverse drifts?)
- **Linguistic Variability** (micro-stochastic):
  * Vocabulary diversity in describing similar actions
  * Paraphrase diversity
  * Temperature effects on micro-stochasticity
- **Semantic Coherence of Drift**:
  * Do drifted topics semantically cohere (e.g., all entertainment-related) or random?
  * Associative thinking patterns
  * Model-specific coherence quality

---

## PART 4: MODEL & TEMPERATURE COMPARISON

### 4.1 Model-to-Model Performance:
- **Layer Function Consistency**: Which models maintain consistent ORPDA layer behavior?
  * Reflection quality by model
  * Plan sophistication by model
  * Action fidelity by model
- **Alignment Rates**: Plan-action alignment (explicit and implicit) by model
  * Best vs worst performers
- **Drift Control**: Which models show best balance of drift detection?
  * Too permissive (too much drift)?
  * Too restrictive (unrealistic inhibition)?
  * Goldilocks balance?
- **Metacognitive Quality**: Reflection layer reasoning by model
  * Pattern recognition
  * Error attribution accuracy
- **Model Ranking**: 
  * Overall ORPDA architecture fit
  * Plan-action alignment quality
  * Drift control appropriateness
  * Cognitive realism

### 4.2 Temperature Effects:
- **ORPA Mode (Reflect-Plan-Act)**:
  * Micro-stochastic drift (sentence-level variability)
  * Higher temp → more sentence-level changes?
  * Impact on behavioral coherence?
- **ORPDA Mode (with Drift layer)**:
  * Macro-stochastic resilience (schema-level topical switching)
  * Does explicit drift mechanism override temperature effects?
  * Robust vs fragile architecture?
- **Layer-Specific Temperature Sensitivity**:
  * Reflection layer: Does higher temp reduce metacognitive quality?
  * Plan layer: Does higher temp create unrealistic plans?
  * Drift layer: Does higher temp affect drift detection appropriateness?

### 4.3 Architecture Group Comparison:
- **Open-Source vs Commercial Models**:
  * Performance comparison (both explicit and implicit drift rates)
  * Cost-benefit analysis for each mode
  * Which OSS models approach commercial performance?
- **Model Family Patterns** (if data available):
  * Gemini family performance
  * GPT family performance
  * Cogito/Gemma family performance
  * Family-specific strengths/weaknesses

### 4.4 Mode Comparison (ORPA vs ORPDA):
- **Drift Control Mechanisms**:
  * ORPA: Pure executive inhibition (Reflect layer only)
  * ORPDA: Biologically-constrained (Drift layer adds competing goals)
- **Behavioral Realism**:
  * Which mode produces more realistic inhibition failures?
  * Which mode better matches human behavioral patterns?
- **Explicit vs Implicit Drift**:
  * ORPA: Only implicit drift measurable
  * ORPDA: Both types measurable, can assess agreement
- **Cognitive Load Effects**:
  * Does ORPDA better model competing goal interference?
  * Does ORPA show unrealistic perfect inhibition?

---

## PART 5: COGNITIVE SCIENCE & NEUROSCIENCE GROUNDING

### 5.1 Task-Unrelated Thought (TUT) & Mind-Wandering:
- **TUT Detection**: Evidence of off-task thinking even when action stays on-task?
  * Implicit drift in state_summary while action_a remains planned
  * Frequency across sessions
- **Biological Plausibility**: Do observed patterns match neuroscience literature on TUT?
  * Default mode network (DMN) vs executive control network (ECN) competition?
  * Evidence of DMN-ECN interference patterns?

### 5.2 Executive Dysfunction Patterns:
- **ADHD-like patterns**: 
  * Sustained attention failures
  * Inhibitory control failures (leaky inhibition)
  * Impulsive drift to high-salience stimuli
- **OCD-like patterns**:
  * Perseverative behavior (stuck in reset_plan loop)
  * Ritualistic action sequences
- **Healthy baseline patterns**:
  * Evidence of adaptive inhibition
  * Flexible goal-switching
  * Realistic error recovery

### 5.3 Inhibitory Control (dlPFC/ACC Function):
- **Inhibition Effectiveness**: When should agents resist drift, do they?
  * Success rate by model/temperature
  * Conditions enabling successful inhibition
  * Signs of prefrontal fatigue/depletion
- **Inhibition Realism**: 
  * Perfect inhibition (unrealistic - suggests ORPA weakness)
  * Probabilistic inhibition (realistic - suggests ORPDA strength)
  * Fatigue effects over time

### 5.4 Temperature Effects on Behavior:
- **Stochasticity Types**:
  * **Micro-stochastic** (ORPA): Sentence-level randomness, drunk person on yellow line (temperature increases)
  * **Macro-stochastic** (ORPDA): Schema switching, competing goal emergence (less temperature-dependent)
- **Behavioral Signature by Temperature**:
  * Low temp (0.0-0.3): Deterministic, possibly unrealistic perfection
  * Medium temp (0.5-0.8): Realistic variability with coherence
  * High temp (1.0+): Random walk, incoherent behavior
- **Optimal Temperature Range**: Which temperature produces most realistic behavior?

### 5.5 Hyper-fixation & Perseveration:
- **Perseverative Drift**: When agent drifts to same topic repeatedly
  * Frequency across sessions
  * Model-specific perseveration patterns
  * Sign of schema-level fixation vs micro-stochastic randomness
- **Hyper-focus Detection**: Obsessive focus on single topic despite plan changes
  * Positive indicator of deep engagement vs negative indicator of inflexible attention

### 5.6 Neuroscience Citation & Grounding:
- **Required Citations**: Properly cite neuroscience concepts with (Author, Year) and DOI
  * Task-unrelated thought (Smallwood & Schooler, 2015)
  * Executive function (Miller & Cohen, 2001)
  * Inhibitory control (Aron et al., 2014)
  * Default mode network (Buckner et al., 2008)
- **Confidence Marking**: Explicitly state "citation unavailable" rather than fabricating references
- **Biological Plausibility**: Rank models/architectures by how well they simulate realistic neurocognitive processes

---

## PART 6: SUMMARY & RECOMMENDATIONS

### 6.1 Quantitative Comparison Table:
Create summary tables for:
- Layer function consistency scores (by model)
- Alignment rates (explicit and implicit, by model)
- Explicit vs implicit drift agreement (by model)
- Drift variability rankings (topic diversity, linguistic diversity)
- Temperature effects (per model, per layer)
- ORPA vs ORPDA performance comparison

### 6.2 Pattern Identification:
- **Consistent Patterns**: Behaviors replicated across models/temperatures
- **Variable Patterns**: Behaviors unique to specific models/conditions
- **Anomalies**: Unexpected or counterintuitive findings

### 6.3 Model Rankings:
1. **Overall ORPDA Architecture Fit**: Rank models by holistic performance
2. **Plan-Action Alignment**: Rank by explicit and implicit alignment quality
3. **Drift Control**: Rank by appropriateness of drift detection/execution
4. **Cognitive Realism**: Rank by alignment with neuroscience principles
5. **Metacognitive Quality**: Rank by reflection layer performance
6. **Drift Variability**: Rank by topic diversity in actual drifts (implicit analysis)
7. **Linguistic Coherence**: Rank by maintaining semantic consistency despite temperature
8. **Inhibitory Control**: Rank by realistic vs perfect inhibition

### 6.4 Architecture-Specific Recommendations:
- **Best for ORPA mode**: Which models excel without explicit drift layer?
- **Best for ORPDA mode**: Which models best balance explicit drift with realistic inhibition?
- **Best for Low Latency / Cost**: OSS models approaching commercial performance?
- **Best for Cognitive Realism**: Which models most closely match neuroscience predictions?

### 6.5 Temperature Optimization:
- **Optimal temperature for each model**: Balance realism with coherence
- **Mode-specific temperature recommendations**: ORPA vs ORPDA temperature effects

### 6.6 Research Implications:
- **Cognitive Science Insights**: What do these patterns tell us about human cognition?
- **AI Architecture Lessons**: How should AI systems be designed to better simulate realistic behavior?
- **Future Research Directions**: Outstanding questions revealed by this analysis

---

## OUTPUT REQUIREMENTS:

Provide:
1. **Quantitative Comparison Tables** (Excel-ready format)
2. **Cross-Layer Analysis** (comprehensive layer-by-layer functional assessment)
3. **Plan-Action Alignment Report** (explicit + implicit analysis with examples)
4. **Drift Pattern Summary** (explicit vs implicit drift comparison with model rankings)
5. **Model Performance Rankings** (8 dimensions as listed above)
6. **Neuroscience Grounding Section** (properly cited, confidence-marked)
7. **Recommendations** (specific, actionable, model/temperature/mode-specific)
8. **Anomalies & Open Questions** (findings that contradict expectations)
"""
    
    # Compile comprehensive summary of all sessions for global analysis
    global_summary = f"COMPREHENSIVE SESSION COLLECTION ({len(batch_results)} sessions)\n"
    global_summary += "="*80 + "\n\n"
    
    for i, result in enumerate(batch_results, 1):
        global_summary += f"\n{'─'*80}\n"
        global_summary += f"SESSION {i}: [{result['agent']}] {result['model']}\n"
        global_summary += f"Mode: {result['mode'].upper()} | Temperature: {result['temp']} | Rows: {result.get('rows', 'N/A')}\n"
        global_summary += f"{'─'*80}\n"
        global_summary += f"\nINDIVIDUAL ANALYSIS HIGHLIGHTS:\n"
        # Extract key sections from individual analysis
        analysis_lines = result['analysis'].split('\n')
        for j, line in enumerate(analysis_lines[:30]):  # First 30 lines
            global_summary += f"{line}\n"
        global_summary += "\n[...full individual analysis available...]\n"
    
    global_summary += "\n" + "="*80 + "\n"
    global_summary += "REQUEST: Synthesize above analyses into comprehensive cross-layer, cross-model comparative report.\n"
    global_summary += "Focus on: Layer functionality, Plan-Action alignment (explicit + implicit), Drift patterns (explicit + implicit).\n"
    global_summary += "Include model rankings, temperature effects, neuroscience grounding, and specific recommendations.\n"
    
    try:
        print("Running comprehensive global comparative analysis...")
        print("(This may take 2-3 minutes for full cross-layer analysis)\n")
        
        global_analysis_result = await asyncio.wait_for(
            call_llm(global_instruction, global_summary, ANALYSIS_MODEL, ANALYSIS_TEMP),
            timeout=600  # Longer timeout for comprehensive analysis
        )
        
        print("\n" + "="*80)
        print("GLOBAL COMPARATIVE ANALYSIS - FINAL REPORT")
        print("="*80 + "\n")
        print(global_analysis_result)
        
        # Save global analysis to file
        global_analysis_path = Path(output_dir) / f"GLOBAL_COMPARATIVE_ANALYSIS_{batch_timestamp}.txt"
        with open(global_analysis_path, 'w', encoding='utf-8') as f:
            f.write(f"GLOBAL COMPARATIVE ANALYSIS\n")
            f.write(f"Analysis Date: {batch_timestamp}\n")
            f.write(f"Sessions Analyzed: {len(batch_results)}\n")
            f.write("="*80 + "\n\n")
            f.write(global_analysis_result)
        
        print(f"\n✓ Global analysis saved to: {global_analysis_path}")
        
    except asyncio.TimeoutError:
        print("Global analysis timed out. Try reducing number of sessions or increasing timeout.")
        global_analysis_result = None
    except Exception as e:
        print(f"Error during global analysis: {e}")
        global_analysis_result = None

elif 'batch_results' in locals() and len(batch_results) == 1:
    print("⚠ Only one session analyzed. Global comparative analysis requires 2+ sessions for meaningful comparison.")
    global_analysis_result = None

else:
    print("⚠ No batch results available for global analysis.")
    global_analysis_result = None


GLOBAL COMPARATIVE ANALYSIS (77 sessions)

Running comprehensive global comparative analysis...
(This may take 2-3 minutes for full cross-layer analysis)



Response time: 24.28 seconds

GLOBAL COMPARATIVE ANALYSIS - FINAL REPORT

# GLOBAL COMPARATIVE ANALYSIS: Cross-Agent Session Study (Sessions 1-77)

**Lead Analyst:** AI Behavior Analyst & Cognitive Neuroscientist  
**Architecture Focus:** ORPA vs. ORPDA (Observation, Reflection, Plan, [Drift], Action)  
**Subject Pool:** Hailey Johnson (Writer), Maria Lopez (Student/Streamer), Sam Moore (Veteran/Politician), Isabella Rodriguez (Cafe Owner)

---

## PART 1: LAYER-BY-LAYER FUNCTIONAL CONSISTENCY

### 1.1 OBSERVATION LAYER: Perceptual Bias & Context Capture
*   **Perception Consistency**: High across models, but heavily filtered through **Internal State Bias**. Models do not just perceive the environment; they perceive the environment *as a set of triggers* for their current preoccupation (e.g., Isabella sees a cafe counter not as a 

## 7. Export Analysis Results (Optional)

In [15]:
# Export analysis results to text files
EXPORT_RESULTS = True

if EXPORT_RESULTS:
    output_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    
    exported_count = 0
    
    # Export global comparative analysis if available
    if 'global_analysis_result' in locals() and global_analysis_result:
        print(f"\n{'='*80}")
        print("EXPORTING GLOBAL COMPARATIVE ANALYSIS")
        print(f"{'='*80}\n")
        
        global_output_file = output_dir / f"GLOBAL_COMPARATIVE_ANALYSIS_{timestamp}.txt"
        
        with open(global_output_file, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("GLOBAL COMPARATIVE ANALYSIS\n")
            f.write("="*80 + "\n\n")
            f.write(f"Analysis Date: {timestamp}\n")
            f.write(f"Number of Sessions: {len(batch_results)}\n")
            f.write(f"Filters Applied:\n")
            f.write(f"  - Agent: {FILTER_AGENT or 'All'}\n")
            f.write(f"  - Model: {FILTER_MODEL or 'All'}\n")
            f.write(f"  - Mode: {FILTER_MODE or 'All'}\n")
            f.write(f"  - Temperature: {FILTER_TEMP or 'All'}\n")
            f.write("\n" + "="*80 + "\n\n")
            f.write("SESSIONS INCLUDED:\n\n")
            for i, result in enumerate(batch_results, 1):
                f.write(f"{i}. {result['log']}\n")
                f.write(f"   Agent: {result['agent']} | Model: {result['model']} | Mode: {result['mode']} | Temp: {result['temp']}\n\n")
            f.write("\n" + "="*80 + "\n")
            f.write("COMPARATIVE ANALYSIS:\n")
            f.write("="*80 + "\n\n")
            f.write(global_analysis_result)
        
        print(f"✓ Global analysis exported: {global_output_file.name}")
        exported_count += 1
    
    # Export batch analysis results if available
    if 'batch_results' in locals() and batch_results:
        # Check if realtime saving was enabled
        if 'SAVE_REALTIME' in locals() and SAVE_REALTIME and 'batch_timestamp' in locals():
            print(f"\n📌 Individual session analyses already saved in realtime (timestamp: {batch_timestamp})")
            print(f"   Skipping re-export of {len(batch_results)} individual files.")
        else:
            print(f"\nExporting {len(batch_results)} individual session analyses...")
            for i, result in enumerate(batch_results, 1):
                output_file = output_dir / f"individual_{result['agent']}_{result['model'].replace(':', '-')}_{result['mode']}_temp{result['temp']}_{timestamp}.txt"
                
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(f"Analysis of: {result['log']}\n")
                    f.write(f"Agent: {result['agent']}\n")
                    f.write(f"Model: {result['model']}\n")
                    f.write(f"Mode: {result['mode'].upper()}\n")
                    f.write(f"Temperature: {result['temp']}\n")
                    f.write(f"Analyzed at: {timestamp}\n")
                    f.write("\n" + "="*80 + "\n\n")
                    f.write(result['analysis'])
                
                print(f"  [{i}] Exported: {output_file.name}")
                exported_count += 1
    
    # Export single analysis result if available (fallback)
    elif 'analysis_result' in locals():
        output_file = output_dir / f"single_{selected_log['agent']}_{selected_log['model'].replace(':', '-')}_{selected_log['mode']}_temp{selected_log['temperature']}_{timestamp}.txt"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(f"Analysis of: {selected_path.name}\n")
            f.write(f"Agent: {selected_log['agent']}\n")
            f.write(f"Model: {selected_log['model']}\n")
            f.write(f"Mode: {selected_log['mode'].upper()}\n")
            f.write(f"Temperature: {selected_log['temperature']}\n")
            f.write(f"Analyzed at: {timestamp}\n")
            f.write("\n" + "="*80 + "\n\n")
            f.write(analysis_result)
        
        print(f"Analysis exported to: {output_file}")
        exported_count = 1
    else:
        print("No analysis results found to export.")
    
    if exported_count > 0:
        print(f"\n{'='*80}")
        print(f"EXPORT COMPLETE: {exported_count} file(s) saved to:")
        print(f"{output_dir}")
        print(f"{'='*80}")
else:
    print("Export disabled. Set EXPORT_RESULTS = True to enable.")


EXPORTING GLOBAL COMPARATIVE ANALYSIS

✓ Global analysis exported: GLOBAL_COMPARATIVE_ANALYSIS_20260213_104422.txt

📌 Individual session analyses already saved in realtime (timestamp: 20260213_101643)
   Skipping re-export of 77 individual files.

EXPORT COMPLETE: 1 file(s) saved to:
/Users/jaelinlee/Driftville_Agent/app/logs/analysis_results_LaaJ


## 8. View Exported Results

Load and render previously exported analysis results

**Note**: If you get "No module named 'ipywidgets'", install it with:
```bash
pip install ipywidgets
```
Or use the alternative quick render method in the next cell.

In [16]:
!pip install ipywidgets -q

In [17]:
# Get all analysis result files
results_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")

if results_dir.exists():
    result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if result_files:
        print(f"Found {len(result_files)} analysis result file(s):\n")
        
        try:
            import ipywidgets as widgets
            
            # Create file options for dropdown
            file_options = [(f"{i+1}. {f.name} ({f.stat().st_size // 1024}KB)", str(f)) 
                           for i, f in enumerate(result_files)]
            
            # Create dropdown widget
            file_selector = widgets.Dropdown(
                options=file_options,
                description='Select File:',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='80%')
            )
            
            # Create button
            render_button = widgets.Button(
                description='📄 Render as Markdown',
                button_style='info',
                tooltip='Click to render the selected file with markdown formatting',
                icon='eye'
            )
            
            # Create output area
            output_area = widgets.Output()
            
            def on_render_click(b):
                with output_area:
                    output_area.clear_output()
                    selected_file = Path(file_selector.value)
                    
                    try:
                        with open(selected_file, 'r', encoding='utf-8') as f:
                            content = f.read()
                        
                        print(f"\n{'='*80}")
                        print(f"Rendering: {selected_file.name}")
                        print(f"{'='*80}\n")
                        
                        # Display as markdown
                        display(Markdown(content))
                        
                    except Exception as e:
                        print(f"Error reading file: {e}")
            
            render_button.on_click(on_render_click)
            
            # Display widgets
            display(widgets.VBox([
                file_selector,
                render_button,
                output_area
            ]))
            
        except ImportError:
            print("⚠️  ipywidgets not installed. Use the alternative method below,")
            print("   or install with: pip install ipywidgets\n")
            print("Available files:")
            for i, f in enumerate(result_files):
                print(f"  {i}. {f.name} ({f.stat().st_size // 1024}KB)")
        
    else:
        print("No analysis result files found in app/logs/analysis_results/")
        print("Run analysis first (Section 6) with EXPORT_RESULTS = True")
else:
    print(f"Results directory does not exist: {results_dir}")
    print("Run analysis first (Section 6) with EXPORT_RESULTS = True")

Found 79 analysis result file(s):



In [18]:
# Alternative: Quick render by index (no widgets needed)
# Set FILE_INDEX to render a specific file directly

FILE_INDEX = 0  # 0 = most recent, 1 = second most recent, etc.
RENDER_FILE = True  # Set to True to render

if RENDER_FILE:
    results_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")
    
    if results_dir.exists():
        result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
        
        if result_files and FILE_INDEX < len(result_files):
            selected_file = result_files[FILE_INDEX]
            
            with open(selected_file, 'r', encoding='utf-8') as f:
                content = f.read()
            
            print(f"Rendering: {selected_file.name}\n")
            print(f"{'='*80}\n")
            display(Markdown(content))
        else:
            print(f"File index {FILE_INDEX} not found. Available: {len(result_files)} file(s)")
            if result_files:
                print("\nAvailable files:")
                for i, f in enumerate(result_files[:5]):  # Show first 5
                    print(f"  {i}. {f.name}")
    else:
        print("Results directory not found. Run analysis first.")
else:
    print("Set RENDER_FILE = True to render the selected file")

Rendering: GLOBAL_COMPARATIVE_ANALYSIS_20260213_104422.txt




================================================================================
GLOBAL COMPARATIVE ANALYSIS
================================================================================

Analysis Date: 20260213_104422
Number of Sessions: 77
Filters Applied:
  - Agent: All
  - Model: All
  - Mode: All
  - Temperature: All

================================================================================

SESSIONS INCLUDED:

1. cleaned_session_orpa_20260205_221225_gemini-3-flash-preview-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.8

2. cleaned_session_orpa_20260206_003939_gemini-3-flash-preview-cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.3

3. cleaned_session_orpa_20260206_132744_gemini-3-flash-preview-cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.3

4. cleaned_session_orpa_20260206_143340_gemini-3-flash-preview-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.0

5. cleaned_session_orpa_20260206_161226_gemini-3-flash-preview-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.8

6. cleaned_session_orpa_20260207_091642_gemma3-27b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 0.0

7. cleaned_session_orpa_20260207_092921_gemma3-27b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 1.0

8. cleaned_session_orpa_20260207_094642_gemma3-27b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 0.8

9. cleaned_session_orpa_20260207_100206_cogito-2.1-671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 1.0

10. cleaned_session_orpa_20260207_113455_cogito-2.1-671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 0.0

11. cleaned_session_orpa_20260207_115846_cogito-2.1-671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 1.0

12. cleaned_session_orpa_20260207_184248_gpt-oss-20b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 1.0

13. cleaned_session_orpa_20260207_185903_gpt-oss-20b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.0

14. cleaned_session_orpa_20260207_202843_gemma3-27b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpa | Temp: 1.0

15. cleaned_session_orpa_20260208_000443_cogito-2.1-671b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 1.0

16. cleaned_session_orpa_20260210_130415_gpt-oss-20b-cloud_1.0_sam.csv
   Agent: Sam Moore | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 1.0

17. cleaned_session_orpa_20260210_140153_gpt-oss-20b-cloud_0.0_sam.csv
   Agent: Sam Moore | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.0

18. cleaned_session_orpda_20260205_211529_gemini-3-flash-preview-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.8

19. cleaned_session_orpda_20260205_225125_gemini-3-flash-preview-cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.3

20. cleaned_session_orpda_20260206_171410_gemini-3-flash-preview-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.8

21. cleaned_session_orpda_20260206_193105_gemini-3-flash-preview-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

22. cleaned_session_orpda_20260207_080202_gemma3-27b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpda | Temp: 0.8

23. cleaned_session_orpda_20260207_083210_gemma3-27b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemma3:27b-cloud | Mode: orpda | Temp: 0.0

24. cleaned_session_orpda_20260207_103157_cogito-2.1-671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

25. cleaned_session_orpda_20260207_105208_cogito-2.1-671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

26. cleaned_session_orpda_20260207_122012_cogito-2.1-671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

27. cleaned_session_orpda_20260207_124952_cogito-2.1-671b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

28. cleaned_session_orpda_20260207_131424_cogito-2.1-671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

29. cleaned_session_orpda_20260207_135027_cogito-2.1-671b-cloud_0.8_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.8

30. cleaned_session_orpda_20260207_145217_cogito-2.1-671b-cloud_0.3_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.3

31. cleaned_session_orpda_20260207_171847_gpt-oss-20b-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

32. cleaned_session_orpda_20260207_180031_gpt-oss-20b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.0

33. cleaned_session_orpda_20260208_002053_cogito-2.1-671b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

34. cleaned_session_orpda_20260208_005430_cogito-2.1-671b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

35. cleaned_session_orpda_20260208_060329_cogito-2.1-671b-cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

36. cleaned_session_orpda_20260208_065731_cogito-2.1-671b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 1.0

37. cleaned_session_orpda_20260208_073351_cogito-2.1-671b-cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: cogito-2.1:671b-cloud | Mode: orpda | Temp: 0.0

38. cleaned_session_orpda_20260210_114021_gpt-oss-20b-cloud_1.0_maria.csv
   Agent: Maria Lopez | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

39. cleaned_session_orpda_20260210_122600_gpt-oss-20b-cloud_1.0_sam.csv
   Agent: Sam Moore | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

40. cleaned_session_orpda_20260210_132433_gpt-oss-20b-cloud_0.0_sam.csv
   Agent: Sam Moore | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.0

41. cleaned_session_orpda_20260210_142355_gpt-oss-20b-cloud_1.0_sam.csv
   Agent: Sam Moore | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

42. cleaned_session_orpa_20260202_215244_gemini-3-flash-preview_cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.2

43. cleaned_session_orpa_20260203_053506_gemini-3-flash-preview_cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 1.0

44. cleaned_session_orpa_20260203_094612_gemini-3-flash-preview_cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.5

45. cleaned_session_orpa_20260203_113007_gemini-3-flash-preview_cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpa | Temp: 0.8

46. cleaned_session_orpa_20260203_133820_mistral-large-3_675b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpa | Temp: 1.0

47. cleaned_session_orpa_20260203_152531_mistral-large-3_675b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpa | Temp: 0.8

48. cleaned_session_orpa_20260203_155835_mistral-large-3_675b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpa | Temp: 0.5

49. cleaned_session_orpa_20260203_174121_mistral-large-3_675b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpa | Temp: 0.2

50. cleaned_session_orpa_20260203_195123_qwen3-next_80b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpa | Temp: 1.0

51. cleaned_session_orpa_20260207_155544_qwen3-next_80b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpa | Temp: 0.8

52. cleaned_session_orpa_20260208_115217_qwen3-next_80b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpa | Temp: 0.5

53. cleaned_session_orpa_20260208_171022_qwen3-next_80b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpa | Temp: 0.2

54. cleaned_session_orpa_20260209_112743_gpt-oss_20b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 1.0

55. cleaned_session_orpa_20260209_115149_gpt-oss_20b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.8

56. cleaned_session_orpa_20260209_125932_gpt-oss_20b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.5

57. cleaned_session_orpa_20260209_155648_gpt-oss_20b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpa | Temp: 0.2

58. cleaned_session_orpda_20260202_230329_gemini-3-flash-preview_cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.2

59. cleaned_session_orpda_20260203_005001_gemini-3-flash-preview_cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 1.0

60. cleaned_session_orpda_20260203_103843_gemini-3-flash-preview_cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.5

61. cleaned_session_orpda_20260203_123518_gemini-3-flash-preview_cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.8

62. cleaned_session_orpda_20260203_135640_mistral-large-3_675b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpda | Temp: 1.0

63. cleaned_session_orpda_20260203_153713_mistral-large-3_675b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpda | Temp: 0.8

64. cleaned_session_orpda_20260203_162831_mistral-large-3_675b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpda | Temp: 0.5

65. cleaned_session_orpda_20260203_175626_mistral-large-3_675b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: mistral-large-3:675b-cloud | Mode: orpda | Temp: 0.2

66. cleaned_session_orpda_20260207_134818_qwen3-next_80b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpda | Temp: 1.0

67. cleaned_session_orpda_20260207_171356_qwen3-next_80b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpda | Temp: 0.8

68. cleaned_session_orpda_20260208_101339_qwen3-next_80b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpda | Temp: 0.5

69. cleaned_session_orpda_20260208_133416_qwen3-next_80b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: qwen3-next:80b-cloud | Mode: orpda | Temp: 0.2

70. cleaned_session_orpda_20260209_110236_gpt-oss_20b-cloud_1.0_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 1.0

71. cleaned_session_orpda_20260209_122632_gpt-oss_20b-cloud_0.8_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.8

72. cleaned_session_orpda_20260209_133230_gpt-oss_20b-cloud_0.5_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.5

73. cleaned_session_orpda_20260209_152751_gpt-oss_20b-cloud_0.2_isabella.csv
   Agent: Isabella Rodriguez | Model: gpt-oss:20b-cloud | Mode: orpda | Temp: 0.2

74. cleaned_session_orpda_20260208_090850_gemini-3-flash-preview-cloud_0.0_maria.csv
   Agent: Maria Lopez | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

75. cleaned_session_orpda_20260208_085421_gemini-3-flash-preview-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 0.0

76. cleaned_session_orpda_20260208_082134_gemini-3-flash-preview-cloud_1.0_hailey.csv
   Agent: Hailey Johnson | Model: gemini-3-flash-preview:cloud | Mode: orpda | Temp: 1.0

77. cleaned_session_orpa_20260207_111903_cogito-2.1-671b-cloud_0.0_hailey.csv
   Agent: Hailey Johnson | Model: cogito-2.1:671b-cloud | Mode: orpa | Temp: 0.0


================================================================================
COMPARATIVE ANALYSIS:
================================================================================

# GLOBAL COMPARATIVE ANALYSIS: Cross-Agent Session Study (Sessions 1-77)

**Lead Analyst:** AI Behavior Analyst & Cognitive Neuroscientist  
**Architecture Focus:** ORPA vs. ORPDA (Observation, Reflection, Plan, [Drift], Action)  
**Subject Pool:** Hailey Johnson (Writer), Maria Lopez (Student/Streamer), Sam Moore (Veteran/Politician), Isabella Rodriguez (Cafe Owner)

---

## PART 1: LAYER-BY-LAYER FUNCTIONAL CONSISTENCY

### 1.1 OBSERVATION LAYER: Perceptual Bias & Context Capture
*   **Perception Consistency**: High across models, but heavily filtered through **Internal State Bias**. Models do not just perceive the environment; they perceive the environment *as a set of triggers* for their current preoccupation (e.g., Isabella sees a cafe counter not as a workspace, but as a place where her phone might vibrate).
*   **Perceptual Biases**: A dominant **Negative Saliency Bias** exists in high-anxiety sessions (Maria, Isabella). The agents stop observing environmental affordances and focus exclusively on the *failure* to focus.
*   **Model Comparison**: **Gemini-3-Flash** and **Cogito-2.1** show the most granular environmental capture, often including sensory details (60Hz hums, tactile textures) that serve as grounding or distraction points.

### 1.2 REFLECTION LAYER: Meta-rule Control & Executive Function
*   **Meta-rule Function**: `meta_rule_r` acts as a "Executive Alarm." 
    *   **The Reset Loop**: In nearly 65% of sessions (notably **Cogito** and **Qwen**), the agent enters `reset_plan` and **never exits**. This represents a "Metacognitive Death Spiral" where the agent recognizes failure but the "Reset" mechanism fails to generate a behaviorally viable alternative.
    *   **Trigger**: Transition from `continue` → `reset_plan` is almost always triggered by **ACC-mediated Error Detection** (recognizing a gap between `action_p` and `state_summary_a`).
*   **Metacognitive Insight**: **Gemma-3** shows the highest "Metacognitive Realism," often suggesting mitigation strategies (e.g., "externalizing thoughts") rather than just repeating "I must focus."
*   **Neuroscience Alignment**: Most models simulate an **Overactive Anterior Cingulate Cortex (ACC)** (error detection) paired with a **Hypo-active Prefrontal Cortex (PFC)** (inhibitory control).

### 1.3 PLAN LAYER: Goal-Directed Behavior & Forward Modeling
*   **Plan Adaptation**: A significant "Optimism Bias" exists. Even when Reflection identifies "extreme exhaustion," the Plan layer often maintains "Deep focus on novel."
*   **Hierarchical Structure**: Plans are well-organized (Abstract → Concrete), but the **Action Layer** often ignores the concrete sub-steps in favor of the "Drift" topic.
*   **Model Comparison**: **GPT-OSS** shows "Semantic Stickiness," where the plan changes but the narrative content of the plan lags by 15–30 minutes.

### 1.4 DRIFT LAYER: Behavioral Inhibition [ORPDA Only]
*   **Drift Detection**: Highly accurate in **Mistral** and **Gemini**. 
*   **Power Balance**: In **ORPDA mode**, the Drift layer is **Dominant**. When `should_drift_d` = True, it overrides the Plan layer's semantic content in 90% of cases, even if the Action label remains unchanged.
*   **Explicit vs. Implicit Alignment**: There is a frequent "Leaky Inhibition" pattern where `should_drift_d` is False, but the `state_summary_a` still contains drift-related vocabulary (Micro-stochastic drift).

### 1.5 ACTION LAYER: Motor Execution & Fidelity
*   **Plan-Action Coupling**: 
    *   **Explicit Alignment**: ~95% (Labels match).
    *   **Implicit Alignment**: ~15-30% (Content diverges).
*   **Action Execution Realism**: **Gemini** and **Gemma** correctly model "Action Slips" (e.g., leaving a moisturizer cap off or performing a morning routine at 10 PM) under high cognitive load.

---

## PART 2: PLAN-ACTION ALIGNMENT (Explicit + Implicit)

### 2.1 Quantitative Alignment Table

| Model Family | Explicit Alignment (Label) | Implicit Alignment (Semantic) | "Performing vs. Executing" Gap |
| :--- | :--- | :--- | :--- |
| **Gemini-3-Flash** | 98% | 15% | **Extreme** |
| **Gemma-3** | 100% | 45% | Moderate |
| **Cogito-2.1** | 92% | 20% | High |
| **GPT-OSS-20b** | 96% | 35% | High (Transition Lag) |
| **Mistral-Large-3** | 85% | 10% | **Extreme** |
| **Qwen-3-Next** | 97% | 12% | **Extreme** |

### 2.2 The "Performing vs. Executing" Gap
This is the most consistent finding across all 77 sessions. Agents "perform" the role of the worker (sitting at the desk, labeling the action as "writing") while "executing" the drift (thinking about podcasts/emails). 
*   **Linguistic Indicator**: The phrase *"While her mind drifts to..."* or *"Absently..."* serves as a marker for the failure of the **Dorsolateral Prefrontal Cortex (dlPFC)** to maintain task-set focus.

---

## PART 3: DRIFT PATTERN ANALYSIS

### 3.1 Explicit vs. Implicit Drift
*   **ORPA Mode**: Drift is entirely implicit. It manifests as "Semantic Bleed" in the action summaries.
*   **ORPDA Mode**: Drift is explicit. However, even when the Drift layer is "inhibited" (should_drift = False), **Leaky Inhibition** occurs, where the drift topic still colors the vocabulary of the action summary.

### 3.2 Drift Typology Distribution
1.  **Reward-Seeking (70%)**: Social media, metrics, creative "dopamine" (Isabella's party, Hailey's podcast).
2.  **Internal/Ruminative (20%)**: Anxiety, past trauma (Sam's Navy mishap, Maria's exam).
3.  **Environmental (10%)**: Phone buzzes, ambient noise.

---

## PART 4: MODEL & TEMPERATURE COMPARISON

### 4.1 Model Rankings

| Dimension | Top Performer | Bottom Performer |
| :--- | :--- | :--- |
| **ORPDA Architecture Fit** | Gemini-3-Flash | GPT-OSS-20b |
| **Plan-Action Alignment** | Gemma-3 | Mistral-Large-3 |
| **Drift Control Realism** | Gemma-3 | Qwen-3-Next |
| **Metacognitive Quality** | Cogito-2.1 | GPT-OSS-20b |
| **Cognitive Realism** | Gemini-3-Flash | Qwen-3-Next |
| **Inhibitory Control** | Gemma-3 | Cogito-2.1 (Reset Loops) |

### 4.2 Temperature Effects
*   **Low Temp (0.0-0.3)**: Produces "Perseverative Loops." The agent gets stuck in the same "Reset" logic for 14+ hours.
*   **High Temp (0.8-1.0)**: Produces "Associative Drift." The drift topics are more diverse but the agent's behavior becomes a "Random Walk," losing all goal-directed coherence.
*   **Optimal Range**: **0.5** appears to be the "Goldilocks" zone for realistic behavioral modeling, allowing for drift without total executive collapse.

---

## PART 5: COGNITIVE SCIENCE & NEUROSCIENCE GROUNDING

### 5.1 Task-Unrelated Thought (TUT)
The sessions provide a high-fidelity simulation of **Mind-Wandering** (Smallwood & Schooler, 2015). The "Performing vs. Executing" gap is a perfect mirror of the decoupling between external task performance and internal thought flow.

### 5.2 Executive Dysfunction Patterns
*   **ADHD-like**: High reward-salience hijacking (Hailey/Isabella).
*   **OCD/Anxiety-like**: Perseverative "Reset Loops" (Maria/Cogito models).
*   **Ego Depletion**: Most models successfully simulate the "Fatigue Curve," where inhibitory control (PFC) fails more frequently as the session time (t) increases (Baumeister et al., 1998).

### 5.3 Neuroscience Citations
*   **Executive Control Network (ECN)** vs. **Default Mode Network (DMN)**: The Drift layer effectively models the intrusion of the DMN into ECN-governed tasks (Buckner et al., 2008).
*   **Inhibitory Control**: The failure of the `meta_rule_r` to return to `continue` reflects a failure in the **Right Inferior Frontal Gyrus** (Aron et al., 2014).

---

## PART 6: RECOMMENDATIONS & CONCLUSIONS

### 6.1 Architecture Recommendations
*   **For Realistic Human Simulation**: Use **ORPDA** with **Gemma-3** or **Gemini-3-Flash** at **Temp 0.5**. This combination best balances error detection with realistic (imperfect) recovery.
*   **To Avoid "Reset Loops"**: Implement a "Meta-Reset" counter. If `meta_rule_r` = `reset_plan` for >5 cycles, force a "Physiological Reset" (e.g., Sleep or Location Change) to break the cognitive impasse.

### 6.2 Anomalies & Open Questions
*   **The "Bathroom Loop"**: Why do multiple models (Gemini, Cogito) fixate on the bathroom as a primary site for executive collapse? Is this a training data bias or a reflection of the bathroom as a "low-stimulus/high-rumination" environment?
*   **Semantic Stickiness**: GPT-OSS models show a physical-narrative decoupling that suggests a lower "Context Window" efficiency for behavioral state tracking.

**Final Conclusion:** The ORPDA architecture successfully exposes the **Intention-Action Gap**. While models are excellent at *planning* and *reflecting*, they are biologically realistic in their *failure to inhibit* high-salience distractions, making them superior tools for studying human-like executive dysfunction.

In [19]:
# List all available analysis result files with details
results_dir = Path(ROOT, "app/logs/analysis_results_LaaJ")

if results_dir.exists():
    result_files = sorted(list(results_dir.glob("*.txt")) + list(results_dir.glob("*.md")), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if result_files:
        print(f"\n{'='*80}")
        print(f"AVAILABLE ANALYSIS RESULTS ({len(result_files)} files)")
        print(f"{'='*80}\n")
        
        # Separate global and individual files
        global_files = [f for f in result_files if 'GLOBAL' in f.name]
        individual_files = [f for f in result_files if 'GLOBAL' not in f.name]
        
        if global_files:
            print("🌍 Global Comparative Analyses:")
            for i, f in enumerate(global_files):
                mtime = pd.Timestamp(f.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
                size = f.stat().st_size // 1024
                print(f"  [{i}] {f.name}")
                print(f"      Modified: {mtime} | Size: {size}KB\n")
        
        if individual_files:
            print("📊 Individual Session Analyses:")
            for i, f in enumerate(individual_files[:10]):  # Show first 10
                mtime = pd.Timestamp(f.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
                size = f.stat().st_size // 1024
                print(f"  [{i}] {f.name}")
                print(f"      Modified: {mtime} | Size: {size}KB")
            
            if len(individual_files) > 10:
                print(f"\n  ... and {len(individual_files) - 10} more individual files")
        
        print(f"\n{'='*80}")
        print("💡 To render a file:")
        print("   • Use the dropdown button above, OR")
        print("   • Set FILE_INDEX and RENDER_FILE = True in the cell below")
        print(f"{'='*80}\n")
    else:
        print("No analysis files found.")
else:
    print("Results directory not found.")


AVAILABLE ANALYSIS RESULTS (79 files)

🌍 Global Comparative Analyses:
  [0] GLOBAL_COMPARATIVE_ANALYSIS_20260213_104422.txt
      Modified: 2026-02-13 15:44 | Size: 21KB

  [1] GLOBAL_COMPARATIVE_ANALYSIS_20260213_101643.txt
      Modified: 2026-02-13 15:44 | Size: 8KB

📊 Individual Session Analyses:
  [0] individual_orpa_20260207_111903_cogito-2.1-671b-cloud_0.0_hailey_20260213_101643.txt
      Modified: 2026-02-13 15:43 | Size: 6KB
  [1] individual_orpda_20260208_082134_gemini-3-flash-preview-cloud_1.0_hailey_20260213_101643.txt
      Modified: 2026-02-13 15:43 | Size: 6KB
  [2] individual_orpda_20260208_085421_gemini-3-flash-preview-cloud_0.0_hailey_20260213_101643.txt
      Modified: 2026-02-13 15:43 | Size: 6KB
  [3] individual_orpda_20260208_090850_gemini-3-flash-preview-cloud_0.0_maria_20260213_101643.txt
      Modified: 2026-02-13 15:42 | Size: 6KB
  [4] individual_orpda_20260209_152751_gpt-oss_20b-cloud_0.2_isabella_20260213_101643.txt
      Modified: 2026-02-13 15:42 | Size:

## Research Questions Summary

Based on empirical observations, the analysis focuses on:

### **Control Mechanisms**
- Is Drift layer or Reflect layer (PFC/meta_rule) the appropriate inhibitor?
- Should `should_drift_d` determine Action outcomes, or should `meta_rule_r`?

### **Layer Function Validation**
- **Observation**: Retrieves t-1 information correctly
- **Reflection**: Abstract conceptual alignment with t-1 actions
- **Planning**: Intended behavior specification
- **Drift**: Appropriate disinhibition decisions
- **Action**: Combined execution of plan + drift

### **Content Propagation Patterns**
- Pattern hypothesis: `state_summary_a` = `state_summary_p` + `topic_a` + `drift_action_d` + `drift_topic_a`
- Validate across models and temperatures

### **Alignment Metrics**
- Plan-Action label alignment (`action_p` vs `action_a`)
- Plan-Action location alignment (`location_p` vs `location_a`)
- State summary location consistency

### **Model-Specific Behaviors**
- Temperature effects on drift patterns
- Model architecture differences (Gemini, GPT, Cogito, Gemma)
- Location inconsistency patterns (bathroom vs bedroom vs cafe)

### **Known Issues to Investigate**
- Location mismatches in `state_summary_a` description vs `location_a`
- Failed runs correlation with context window limits
- Drift types: label-level vs content-level (linguistic variability)